---
<font color='Blue' size="4">
F37.206 컴퓨팅 탐색: 실생활에서 활용하기(Exploring Computing: Applications in Everyday Life)</font>

---


# Chapter 14. 데이터 기반 보고서 생성 자동화

<div style="background-color: #f5fff5; padding: 10px; border-radius: 5px; color: #000000;">

<font size=5> <strong> <mark style="background-color: #f5fff5;">✅ 학습목표와 기대효과 </mark></strong></font>

<mark style="background-color: #f5fff5;">
🔹 학습목표<br>
  <ul><li>yfinance와 Pandas를 활용하여 주식 데이터를 조회·분석해보자. </li>
  <li>Matplotlib으로 그래프를 시각화하고 Excel·PDF 형태로 결과물을 생성해보자.</li>
  <li>Streamlit으로 간단한 데이터 조회 웹앱을 만들고 활용해보자. </li></ul> 
🔹 기대효과<br>
  <ul><li>데이터 수집부터 분석·시각화·보고서 생성까지 전체 흐름을 경험함으로써 실무형 데이터 처리 능력이 향상된다.</li>
  <li>사용자 입력 기반의 데이터 분석 프로그램과 웹 대시보드 제작 역량을 갖출 수 있다. </li></ul></mark>
</div>

## <div style="background-color:rgba(208, 205, 208, 1); padding: 10px; border-radius: 5px;"><mark style="background-color: rgba(208, 205, 208, 1);">**자동화 보고 시스템**</mark></div>

- 데이터 자동화 보고 시스템은 다양한 데이터 소스를 기반으로 정보를 수집, 처리, 분석하고, 이를 정해진 형식으로 자동 생성하여 보고하는 시스템을 말한다. 
- 사용자의 개입 없이 주기적으로 혹은 실시간으로 보고서를 작성할 수 있으며, 반복적이고 수작업이 필요한 보고 업무를 자동화하는 것이 핵심이다.
- 이번 시간에는 관심 있는 주식 정보를 자동으로 수집하고, 시각화와 보고서를 Excel과 PDF로 저장하는 스마트 주식 분석기 프로그램을 만들어보자.

## <div style="background-color:rgba(208, 205, 208, 1); padding: 10px; border-radius: 5px;"><mark style="background-color: rgba(208, 205, 208, 1);">**필요 모듈 설치**</mark></div>

- 먼저, 스마트 주식 분석기 구현을 위해 필요한 모듈들을 설치해보자.
    - `yfinance` : 야후 파이낸스(Yahoo Finance)에서 주식 및 금융 데이터를 쉽게 가져오는 라이브러리
    - `openpyxl` : Excel(.xlsx) 파일을 읽고 쓰고 편집할 수 있는 Python 라이브러리
    - `reportlab` : Python에서 PDF 문서를 생성하고 디자인할 수 있는 라이브러리

In [ ]:
!pip install yfinance openpyxl reportlab

## <div style="background-color:rgba(208, 205, 208, 1); padding: 10px; border-radius: 5px;"><mark style="background-color: rgba(208, 205, 208, 1);">**스마트 주식 분석/보고 프로그램**</mark></div>

### 1. 주식 데이터 수집

- 필요 모듈의 설치가 완료되면 모듈을 불러온다.

In [2]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import Workbook
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
import time

- 변수 stocks은 분석하고자 하는 주식들의 심볼(Symbol)을 리스트로 가진다.
    - 심볼: 각 주식을 대표하는 고유 코드
    - 예를 들어, 애플(Apple)의 심볼은 AAPL, 테슬라(Tesla)의 심볼은 TSLA, 삼성전자는 005930.KS처럼 표기된다.
- 변수 period는 가져올 주식 데이터의 기간을 지정한다.
    - 1d → 최근 1일
    - 5d → 최근 5일
    - 1mo → 최근 1개월
    - 3mo → 최근 3개월
    - 6mo → 최근 6개월
    - 1y → 최근 1년
    - 2y → 최근 2년
    - 5y → 최근 5년
    - 10y → 최근 10년
    - ytd → 올해 초부터 오늘까지 (Year To Date)
    - max → 가능한 모든 데이터

In [4]:
stocks = ["AAPL", "TSLA"]  # 분석할 주식 심볼
period = "1mo"  # 데이터 기간 설정 (최근 1개월)


- 주식 데이터는 앞에서 배웠던 BeautifulSoup이나 Selenium을 사용해 웹페이지를 크롤링하는 방식으로 가져오지 않는다.
- yfinance 모듈의 Ticker().history()는 날짜, 시가·고가·저가·종가, 거래량 등 주요 정보를 이미 정리된 형태의 데이터프레임으로 제공한다. 즉, 웹페이지를 분석하는 것이 아니라 가공된 데이터베이스에서 직접 데이터를 받아오는 방식이다.

In [ ]:
print("📊 주식 데이터를 가져오는 중...")
data = {ticker: yf.Ticker(ticker).history(period=period) for ticker in stocks}
data

- 변수 data는 딕셔너리이다. 그렇지만 그 안에 들어가 있는 value는 데이터프레임이다.
- 따라서 딕셔너리의 키를 넣어줬을때 그 값은 데이터프레임 형태로 출력된다.

In [ ]:
data
#data['AAPL']
#data['AAPL'].head()

- 여러 종목의 주가 데이터를 하나의 데이터프레임으로 합치면서, 각 행이 어떤 주식인지 알 수 있도록 심볼(Symbol) 컬럼을 추가하였다.

In [14]:
df = pd.concat([data[ticker].assign(Symbol=ticker) for ticker in stocks])
df.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits,Symbol
Date,,,,,,,,
2025-10-29 00:00:00-04:00,269.019206,271.147148,266.851294,269.438812,51086700,0.0,0.0,AAPL
2025-10-30 00:00:00-04:00,271.726571,273.874513,268.219991,271.137146,69886500,0.0,0.0,AAPL
2025-10-31 00:00:00-04:00,276.721738,277.051436,268.899335,270.108154,86167100,0.0,0.0,AAPL
2025-11-03 00:00:00-05:00,270.158128,270.587704,265.992153,268.789429,50194600,0.0,0.0,AAPL
2025-11-04 00:00:00-05:00,268.070107,271.227050,267.360803,269.778473,49274800,0.0,0.0,AAPL


- DatetimeIndex에서 시간대 정보는 현재 큰 의미가 없기 때문에 시간대(timezone) 정보를 제거하여 날짜 인덱스를 표준 날짜 형식으로 통일시킨다.
    - 2025-12-01 14:00:00+09:00에서 +09:00, UTC인지, KST인지 등의 정보를 제거 

In [ ]:
df.index = df.index.tz_convert(None)  # 2025-12-01 14:00:00+09:00
df.index

- 위 코드를 함수로 변환하여 재사용할 수 있도록 구성하였다.

In [17]:
def get_stock_data(symbol_list, period="1mo"):
    """
    주식 데이터를 가져와서 하나의 DataFrame으로 합쳐 반환하는 함수.

    Parameters:
    - stocks: list, 분석할 주식 심볼 리스트
    - period: str, 데이터 기간 (예: '1mo', '3mo', '1y')

    Returns:
    - df: pandas DataFrame, 모든 주식 데이터를 합친 DataFrame
    """
    print("📊 주식 데이터를 가져오는 중...")

    data = {ticker: yf.Ticker(ticker).history(period=period) for ticker in symbol_list}
    df = pd.concat([data[ticker].assign(Symbol=ticker) for ticker in symbol_list])
    if isinstance(df.index, pd.DatetimeIndex): #일부 티커에는 타임존이 없는 경우도 있음
        if df.index.tz is not None:
            df.index = df.index.tz_convert(None)

    return df, data
symbol_list = ["AAPL", "TSLA"]
df, data = get_stock_data(symbol_list, period="1mo")
print(df.head())

📊 주식 데이터를 가져오는 중...
                           Open        High         Low       Close    Volume  \
Date                                                                            
2025-10-29 04:00:00  269.019206  271.147148  266.851294  269.438812  51086700   
2025-10-30 04:00:00  271.726571  273.874513  268.219991  271.137146  69886500   
2025-10-31 04:00:00  276.721738  277.051436  268.899335  270.108154  86167100   
2025-11-03 05:00:00  270.158128  270.587704  265.992153  268.789429  50194600   
2025-11-04 05:00:00  268.070107  271.227050  267.360803  269.778473  49274800   

                     Dividends  Stock Splits Symbol  
Date                                                 
2025-10-29 04:00:00        0.0           0.0   AAPL  
2025-10-30 04:00:00        0.0           0.0   AAPL  
2025-10-31 04:00:00        0.0           0.0   AAPL  
2025-11-03 05:00:00        0.0           0.0   AAPL  
2025-11-04 05:00:00        0.0           0.0   AAPL  


### 2. 주식 시각화

In [ ]:
def plot_and_save_chart(symbol_list, data):
    chart_filename = "stock_chart.png"
    print("📈 주가 시각화를 생성하는 중...")
    plt.figure(figsize=(12, 6))
    for ticker in symbol_list:
        stock_data = data[ticker]
        stock_data.index = stock_data.index.tz_localize(None)
        plt.plot(stock_data.index, stock_data["Close"], label=ticker)

    plt.title("Stock Closing Prices (Last 1 Month)")
    plt.xlabel("Date")
    plt.ylabel("Price (USD)")
    plt.legend()
    plt.grid()
    plt.xticks(rotation=45)
    # 차트 이미지 저장
    plt.savefig(chart_filename)
    print(f"주가 그래프 저장 완료: {chart_filename}")
    plt.show()
    return chart_filename


chart_filename = plot_and_save_chart(symbol_list, data)

### 3. Excel 파일로 저장

In [ ]:
def save_excel(df):
    # Excel 저장
    excel_filename = "stock_data.xlsx"
    df.to_excel(excel_filename, index=True)
    print(f"✅ Excel 파일 저장 완료: {excel_filename}")

save_excel(df)    

### 4. 보고서 자동 생성

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Image, Spacer
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.cidfonts import UnicodeCIDFont
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter

def create_pdf_report(pdf_filename, chart_filename):
    
    # 한글 폰트 등록 (필수)
    pdfmetrics.registerFont(UnicodeCIDFont("HYSMyeongJo-Medium"))

    doc = SimpleDocTemplate(pdf_filename, pagesize=letter)

    styles = getSampleStyleSheet()

    # 스타일에 한글 폰트 적용
    style_normal = styles["Normal"]
    style_normal.fontName = "HYSMyeongJo-Medium"
    style_title = styles["Title"]
    style_title.fontName = "HYSMyeongJo-Medium"

    summary_text = f"""
    최근 1개월간 {symbol_list} 주식 데이터를 분석한 결과입니다.
    주요 데이터는 Excel에 저장되었습니다.
    아래 그래프는 주식 종가(Closing Price)의 변화를 보여줍니다.
    """    

    report = []
    report.append(Paragraph("주식 시장 보고서", style_title))
    report.append(Spacer(1, 20)) #레이아웃에서 공간(space)을 추가
    report.append(Paragraph(summary_text, style_normal))
    report.append(Spacer(1, 20))

    # 이미지 삽입
    try:
        report.append(Image(chart_filename, width=400, height=200))
    except Exception as e:
        report.append(Paragraph(f"이미지 로딩 오류: {e}", style_normal))

    doc.build(report)
    print(f"PDF 생성 완료: {pdf_filename}")
    print("\n모든 작업이 완료되었습니다! 보고서를 확인하세요.")

# ✅ 보고서 생성 실행

create_pdf_report("stock_report.pdf", chart_filename)

### 5. 메일보내기

1. 구글 계정 보안 페이지 접속
    - 구글 계정 보안 페이지 접속: https://myaccount.google.com/
    - 왼쪽 메뉴에서 보안 및 로그인 클릭

2. 2단계 인증 활성화하기
    - 보안 메뉴에서 `2단계 인증` 항목 찾기
    - `사용 설정` 눌러서 2단계 인증을 켜기(휴대폰 번호 인증 또는 Google Authenticator 설정 필요)

3. 앱 비밀번호(App Password) 생성
    - 다시 보안 → 2단계 인증 페이지로 들어감
    - 아래쪽에 `앱 비밀번호(App passwords)` 메뉴 클릭
    - 구글 계정 비밀번호 한 번 입력
    - 생성 메뉴에서 Mail(메일), Windows 컴퓨터 등 선택

4. 생성된 앱 비밀번호 사용
    - 화면에 16자리 비밀번호가 나타남
    - 앱 비밀번호는 구글 계정 비밀번호와 다름
    - 앱 비밀번호는 특정 기기/앱 전용, 여러 곳에서 재사용 가능
    - 필요 없으면 같은 메뉴에서 삭제 가능
    - 다른 사람에게 절대 공유 금지

※ 앱 비밀번호는 반드시 2단계 인증이 켜져 있어야 생성 가능합니다.

In [ ]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email import encoders

# 보내는 사람/받는 사람
user = "your_email@gmail.com"
password = "앱 비밀번호 또는 실제 비밀번호"
to_email = "받는사람@gmail.com"

# SMTP 서버 설정
smtp_server = "smtp.gmail.com"
smtp_port = 587

# 메일 제목/본문
msg = MIMEMultipart()
msg["From"] = user
msg["To"] = to_email
msg["Subject"] = "테스트 메일"

# 본문 추가
body = "메일 본문 내용입니다."
msg.attach(MIMEText(body, "plain"))

# 첨부파일 추가
filename = "stock_report.pdf"  # 첨부할 파일 이름
with open(filename, "rb") as attachment:
    part = MIMEBase("application", "octet-stream")
    part.set_payload(attachment.read())

# Base64로 인코딩
encoders.encode_base64(part)

# 첨부파일 헤더 설정
part.add_header(
    "Content-Disposition",
    f"attachment; filename= {filename}",
)

msg.attach(part)

# 메일 전송
with smtplib.SMTP(smtp_server, smtp_port) as server:
    server.starttls()
    server.login(user, password)
    server.send_message(msg)

print("메일 발송 완료!")


메일 발송 완료!


## <div style="background-color:rgba(208, 205, 208, 1); padding: 10px; border-radius: 5px;"><mark style="background-color: rgba(208, 205, 208, 1);">**입력 받아 조회하기**</mark></div>

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import requests
import time

def search_symbol(query):
    url = f"https://query1.finance.yahoo.com/v1/finance/search?q={query}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }
    response = requests.get(url, headers=headers)    
    time.sleep(3)
    if response.status_code != 200:
        return []
    result = response.json()
    quotes = result.get("quotes", [])
    return [
    (item["symbol"], item.get("longname", item.get("shortname", "")))
    for item in quotes if "symbol" in item]


query = input("기업명 또는 주식 심볼을 입력하세요: ").strip()
results = search_symbol(query)
print(results)

In [ ]:
def select_symbol(results):
    if len(results) == 0:
        print("⚠ 검색 결과가 없습니다. 종료합니다.")
        return []
    
    elif len(results) == 1:
        selected_symbol = results[0][0]
        print(f"✅ 자동 인식된 심볼: {selected_symbol} ({results[0][1]})")
        return [selected_symbol]   # 리스트로 반환

    else:
        print("여러 기업이 검색되었습니다. 번호를 선택하세요 (여러 개 선택 가능: 1,3 또는 1 3):")
        for i, (sym, name) in enumerate(results):
            print(f"{i+1}. {sym} — {name}")

        raw_input = input("선택 번호 입력: ")

        # 공백 또는 쉼표 단위로 나눠 처리
        choices = raw_input.replace(",", " ").split()
        
        selected_symbols = []
        for ch in choices:
            idx = int(ch) - 1
            if 0 <= idx < len(results):
                selected_symbols.append(results[idx][0])

        print(f"\n### 선택된 심볼 목록: {selected_symbols}")
        return selected_symbols

symbol_list = select_symbol(results)
print(symbol_list)

In [ ]:
df, data = get_stock_data(symbol_list)
print(df.head())

In [ ]:
plot_and_save_chart(symbol_list, data)

In [ ]:
create_pdf_report("stock_report.pdf", chart_filename)

## <div style="background-color:rgba(208, 205, 208, 1); padding: 10px; border-radius: 5px;"><mark style="background-color: rgba(208, 205, 208, 1);">**Streamlit 활용하여 웹에서 조회하기**</mark></div>

In [ ]:
%%writefile Ex_ch14_web1.py
import streamlit as st
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from io import BytesIO
import requests
import time

st.title("📈 주식 데이터 분석 대시보드")

# -------------------------------
# 함수: 기업명 → 심볼 자동 검색
# -------------------------------
def search_symbol(query):
    url = f"https://query1.finance.yahoo.com/v1/finance/search?q={query}"
    response = requests.get(url)
    time.sleep(2)
    if response.status_code != 200:
        return []
    result = response.json()
    quotes = result.get("quotes", [])
    return [(item["symbol"], item.get("longname", item.get("shortname", "")))
            for item in quotes if "symbol" in item]

# -------------------------------
# 사용자 입력
# -------------------------------
query = st.text_input("🔎 기업명 또는 주식 심볼을 입력하세요", "")

selected_symbol = None

if query:
    results = search_symbol(query)

    if len(results) == 0:
        st.warning("검색 결과가 없습니다. 다른 기업명 또는 심볼을 입력해보세요.")
    elif len(results) == 1:
        selected_symbol = results[0][0]
        st.success(f"자동 인식된 심볼: **{selected_symbol}** ({results[0][1]})")
    else:
        options = [f"{sym} — {name}" for sym, name in results]
        choice = st.selectbox("여러 기업이 검색되었습니다. 선택하세요:", options)
        selected_symbol = choice.split(" — ")[0]

# -------------------------------
# 주식 데이터 분석
# -------------------------------
if selected_symbol:
    st.write(f"### 📌 선택된 주식: **{selected_symbol}**")

    stock = yf.Ticker(selected_symbol)
    df = stock.history(period="1mo")
    df.index = df.index.tz_localize(None)

    # -------------------------------
    # 차트 그리기
    # -------------------------------
    st.subheader("📈 주가 차트")

    plt.figure(figsize=(12, 6))
    plt.plot(df.index, df["Close"], label=selected_symbol)
    plt.title(f"{selected_symbol} Closing Prices (1 Month)")
    plt.xlabel("Date")
    plt.ylabel("Price (USD)")
    plt.legend()
    plt.grid()
    plt.xticks(rotation=45)

    st.pyplot(plt)

    # -------------------------------
    # 엑셀 다운로드
    # -------------------------------
    st.subheader("📥 데이터 다운로드")

    excel_buffer = BytesIO()
    df.to_excel(excel_buffer, index=True)
    st.download_button(
        label="📊 엑셀 다운로드",
        data=excel_buffer.getvalue(),
        file_name=f"{selected_symbol}_stock_data.xlsx",
        mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
    )

    # -------------------------------
    # PNG 다운로드
    # -------------------------------
    img_buffer = BytesIO()
    plt.savefig(img_buffer, format="png")
    st.download_button(
        label="📸 차트 이미지 다운로드",
        data=img_buffer.getvalue(),
        file_name=f"{selected_symbol}_chart.png",
        mime="image/png"
    )


## <div style="background-color:rgba(208, 205, 208, 1); padding: 10px; border-radius: 5px;"><mark style="background-color: rgba(208, 205, 208, 1);">**마무리**</mark></div>
이번 수업을 통해 우리는 주식 데이터를 실제로 불러와 분석하고, 그래프 시각화 및 보고서 제작까지 하나의 흐름으로 이어지는 데이터 분석 과정을 직접 경험했다. 또한 Streamlit을 활용해 웹 기반 조회 기능까지 구현함으로써, 배운 내용을 실전 환경에 바로 적용할 수 있는 역량을 갖추게 되었다. 앞으로 다양한 데이터에 응용하며 분석 자동화와 웹 서비스 개발로 확장해보길 기대한다.

---
<font color='Grey' size="4">
F37.206 컴퓨팅 탐색: 실생활에서 활용하기(Exploring Computing: Applications in Everyday Life)</font>

---
서울대학교 학부대학 강의교수 변해선